# Group Members:

*   Goe Jie Ying A23CS0224
*   Nawwarah Auni binti Nazrudin A23CS0143
*   Yasmin Batrisyia Binti Zahiruddin A23CS0201

# 🏠 mudah.my — Team Property Scraper
This python script is designed to scrape a list of properties in mudah.my from every state in Malaysia. The scraper extracts properties-related information including detail page data (description, land title, mortgage, seller info) and stores them into multiple CSV files according to the state.

| Step | What it does |
|------|--------------|
| Step 1 | Install & import packages |
| Step 2 | Set state & offset range |
| Step 3 | Parser functions (listing card + detail page) |
| Step 4 | CSV helper functions |
| Step 5 | Run the scraper |
| Step 6 | Retry failed pages |
| Step 7 | Preview & validate output |
| Step 8 | Download files |
| Step 9 | Team lead merges all files |

**Scrape coordination:**
```
Goe    → johor, kedah, kelantan, melaka, negeri-sembilan
Auni   → selangor, kuala-lumpur, perak, pahang, labuan
Yasmin → sabah, sarawak, perlis, putrajaya, terengganu, penang
```

**Final CSV column order:**
`listing_id, property_type, description, location, state, price_rm, mortgage_est_rm, land_title, size_sqft, bedrooms, bathrooms, tenure, seller_type, seller_name, url, img_url`


## Execution Time

*   Johor - 753 minutes
*   Kedah - 562 minutes
*   Kelatan - 105 minutes
*   Melaka - 597 minutes
*   Negeri Sembilan - 686 minutes




## 📦 Step 1 — Install & Import

In [ ]:
!pip install -q requests beautifulsoup4 lxml pandas openpyxl tqdm

import requests
from bs4 import BeautifulSoup
import pandas as pd
import time, random, re, json, os
from datetime import datetime
from tqdm.notebook import tqdm

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9,ms;q=0.8",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Referer": "https://www.mudah.my/",
    "Connection": "keep-alive",
    "Cache-Control": "no-cache",
}

print('✅ All packages ready.')

✅ All packages ready.


## ⚙️ Step 2 — SET STATE & OFFSET RANGE

In [ ]:
# Manually fill offset
STATE        = "kedah"
START_OFFSET = 174
END_OFFSET   = 250

BASE_URL   = f"https://www.mudah.my/{STATE}/properties-for-sale"
PAGE_STEP  = 1
DELAY_MIN  = 2.5
DELAY_MAX  = 5.0

OUTPUT_CSV = f"mudah_{STATE}.csv"
FAILED_LOG = f"mudah_{STATE}_failed.txt"

TOTAL_PAGES = END_OFFSET - START_OFFSET + 1

print(f"""
{'='*50}
  State         : {STATE}
  Offset range  : {START_OFFSET}  →  {END_OFFSET}
  Total pages   : {TOTAL_PAGES:,}
  Est. records  : ~{TOTAL_PAGES * 43:,}
  Output file   : {OUTPUT_CSV}
  Est. time     : ~{TOTAL_PAGES * 8 / 60:.0f} min (incl. detail pages)
{'='*50}
""")


  State         : kedah
  Offset range  : 174  →  250
  Total pages   : 77
  Est. records  : ~3,311
  Output file   : mudah_kedah.csv
  Est. time     : ~10 min (incl. detail pages)



## 🔧 Step 3 — Parser Functions
Handles both listing card parsing and detail page scraping.

In [ ]:
MALAYSIAN_STATES = [
    'johor', 'kedah', 'kelantan', 'kuala lumpur', 'labuan',
    'melaka', 'negeri sembilan', 'pahang', 'penang', 'perak',
    'perlis', 'putrajaya', 'sabah', 'sarawak', 'selangor', 'terengganu'
]


def fetch_page(session, url, retries=3):
    """Fetch any URL with retry. Returns (soup, error)."""
    for attempt in range(1, retries + 1):
        try:
            resp = session.get(url, headers=HEADERS, timeout=20)
            if resp.status_code == 429:
                wait = 30 * attempt
                print(f"  ⚠️  Rate limited. Waiting {wait}s...")
                time.sleep(wait)
                continue
            resp.raise_for_status()
            return BeautifulSoup(resp.text, 'lxml'), None
        except requests.RequestException as e:
            if attempt < retries:
                time.sleep(5 * attempt)
            else:
                return None, str(e)
    return None, "Max retries exceeded"


def parse_spec_value(text, label):
    m = re.search(rf'([\d,\.]+)\s*{re.escape(label)}', text, re.IGNORECASE)
    return m.group(1).replace(',', '') if m else None


def extract_location_state(raw_location):
    """Split 'Klang, Selangor' → location='Klang', state='Selangor'."""
    if not raw_location:
        return None, STATE.replace('-', ' ').title()
    parts = [p.strip() for p in raw_location.split(',')]
    for ms in MALAYSIAN_STATES:
        if parts[-1].lower() == ms.lower():
            loc = ', '.join(parts[:-1]) if len(parts) > 1 else parts[0]
            return loc, parts[-1]
    # State not found in location text — use STATE variable
    return raw_location, STATE.replace('-', ' ').title()


def find_listing_cards(soup):
    listing_links = soup.find_all('a', href=re.compile(r'-\d+\.htm$'))
    seen, cards = set(), []
    for a in listing_links:
        href = a.get('href', '')
        if href in seen:
            continue
        seen.add(href)
        container = a
        for _ in range(4):
            parent = container.parent
            if parent and parent.name not in ('body', 'html', '[document]', None):
                pt = parent.get_text()
                if 'RM' in pt and ('Bed' in pt or 'sq.ft' in pt or 'Acre' in pt):
                    container = parent
                    break
                container = parent
        cards.append(container)
    return cards


def parse_listing_card(card):
    """Extract fields from a listing card on the search results page."""
    full_text = card.get_text(separator=' ', strip=True)

    link = card.find('a', href=True)
    _url = None
    if link:
        href = link['href']
        _url = href if href.startswith('http') else f"https://www.mudah.my{href}"

    id_m = re.search(r'-(\d+)\.htm', _url or '')
    listing_id = id_m.group(1) if id_m else None

    headings = card.find_all(['h2', 'h3'])
    property_type = headings[0].get_text(strip=True) if headings else None
    raw_location  = headings[1].get_text(strip=True) if len(headings) >= 2 else None
    location, state = extract_location_state(raw_location)

    price_m  = re.search(r'RM\s?([\d,]+)', full_text)
    price_rm = price_m.group(1).replace(',', '') if price_m else None

    size_sqft  = parse_spec_value(full_text, 'sq.ft')
    bedrooms   = parse_spec_value(full_text, 'Bed')
    bathrooms  = parse_spec_value(full_text, 'Bath')
    tenure     = ('Freehold'  if 'Freehold'           in full_text
             else 'Leasehold' if 'Leasehold'          in full_text else None)
    seller_type = ('Private'  if 'Private advertiser'  in full_text
              else 'Agent'    if 'Property agent'      in full_text else 'Unknown')

    return {
        '_url':          _url,
        'listing_id':    listing_id,
        'property_type': property_type,
        'location':      location,
        'state':         state,
        'price_rm':      price_rm,
        'size_sqft':     size_sqft,
        'bedrooms':      bedrooms,
        'bathrooms':     bathrooms,
        'tenure':        tenure,
        'seller_type':   seller_type,
    }


def scrape_detail_page(session, listing_url):
    """Visit one listing detail page and extract enriched fields."""
    detail = {
        'description':     None,
        'mortgage_est_rm': None,
        'land_title':      None,
        'seller_name':     None,
        'img_url':         None,
    }

    soup, error = fetch_page(session, listing_url)
    if soup is None:
        return detail

    full_text = soup.get_text(separator='\n', strip=True)
    lines = [l.strip() for l in full_text.split('\n') if l.strip()]

    # ── Image — from og:image meta tag ────────────────────────────────────
    og_img = soup.find('meta', property='og:image')
    if og_img:
        detail['img_url'] = og_img.get('content')
    else:
        img = soup.find('img', src=re.compile(r'rnudah\.com|cdn\.rnudah'))
        if img:
            detail['img_url'] = img.get('src')

    # ── Description — everything between Description and Property Details ──
    desc_start = full_text.find('Description\n')
    desc_end   = full_text.find('Property Details\n')
    if desc_start == -1:
        desc_start = full_text.find('## Description')
    if desc_end == -1:
        desc_end = full_text.find('## Property Details')
    if desc_start != -1:
        end_pos  = desc_end if desc_end > desc_start else desc_start + 3000
        raw_desc = full_text[desc_start + len('Description\n'):end_pos]
        raw_desc = re.sub(r'show more', '', raw_desc, flags=re.IGNORECASE)
        raw_desc = re.sub(r'Show contact number', '', raw_desc, flags=re.IGNORECASE)
        raw_desc = re.sub(r'\n{3,}', '\n\n', raw_desc)
        detail['description'] = raw_desc.strip()

    # ── Land Title ────────────────────────────────────────────────────────
    for i, line in enumerate(lines):
        if line == 'Land Title' and i + 1 < len(lines):
            detail['land_title'] = lines[i + 1]
            break

    # ── Mortgage Estimation ───────────────────────────────────────────────
    mort_m = re.search(r'RM\s*([\d,]+)\s*\nEstimated monthly installment', full_text)
    if not mort_m:
        mort_m = re.search(r'RM\s*([\d,]+)\s*Estimated monthly installment', full_text)
    if mort_m:
        detail['mortgage_est_rm'] = mort_m.group(1).replace(',', '')

    # ── Seller Name ───────────────────────────────────────────────────────
    seller_link = soup.find('a', href=re.compile(r'/u/seller-\d+'))
    if seller_link:
        detail['seller_name'] = seller_link.get_text(strip=True)

    return detail


print('✅ Parser functions ready.')

✅ Parser functions ready.


## 💾 Step 4 — CSV Helper
Saves records in the exact column order.

In [ ]:
COL_ORDER = [
    'listing_id', 'property_type', 'description', 'location', 'state',
    'price_rm', 'mortgage_est_rm', 'land_title',
    'size_sqft', 'bedrooms', 'bathrooms', 'tenure',
    'seller_type', 'seller_name', 'url', 'img_url'
]


def append_to_csv(records):
    if not records:
        return
    df_new = pd.DataFrame(records)
    for col in COL_ORDER:
        if col not in df_new.columns:
            df_new[col] = None
    df_new = df_new[COL_ORDER]
    write_header = not os.path.exists(OUTPUT_CSV)
    df_new.to_csv(OUTPUT_CSV, mode='a', index=False, header=write_header, encoding='utf-8-sig')


def log_failed(offset, reason):
    with open(FAILED_LOG, 'a') as f:
        f.write(f"offset={offset}\treason={reason}\tat={datetime.utcnow().isoformat()}\n")


def format_duration(seconds):
    """Format seconds into human-readable duration."""
    seconds = int(seconds)
    if seconds < 60:
        return f"{seconds}s"
    elif seconds < 3600:
        m, s = divmod(seconds, 60)
        return f"{m}m {s}s"
    else:
        h, rem = divmod(seconds, 3600)
        m, s   = divmod(rem, 60)
        return f"{h}h {m}m {s}s"


print('✅ CSV helper ready.')

✅ CSV helper ready.


## 🚀 Step 5 — Run the Scraper


In [ ]:
RUN_PAGES = None

scraped_ids = set()
if os.path.exists(OUTPUT_CSV):
    try:
        # We only read the listing_id column to save memory
        existing_df = pd.read_csv(OUTPUT_CSV, usecols=['listing_id'])
        scraped_ids = set(existing_df['listing_id'].astype(str).unique())
        print(f"ℹ️  Found {len(scraped_ids):,} existing records in {OUTPUT_CSV}. These will be skipped.")
    except Exception:
        print(f"ℹ️  Starting fresh or {OUTPUT_CSV} is empty.")

CURRENT_START   = START_OFFSET if START_OFFSET > 0 else 1
all_offsets     = list(range(CURRENT_START, END_OFFSET + 1))
session_offsets = all_offsets[:RUN_PAGES] if RUN_PAGES else all_offsets

print(f"State              : {STATE}")
print(f"Pages this session : {len(session_offsets):,}")
print(f"Offsets            : {session_offsets[0]} → {session_offsets[-1]}")
print(f"Est. time          : ~{len(session_offsets) * 8 / 60:.1f} min\n")

http_session     = requests.Session()
batch_records    = []
batch_page_count = 0
session_total    = 0
skipped_count    = 0
last_done_offset = None
state_start_time = time.time()

try:
    for i, offset in enumerate(session_offsets, 1):
        page_url = f"{BASE_URL}?o={offset}"
        soup, error = fetch_page(http_session, page_url)

        if soup is None:
            print(f"  ❌ Failed offset {offset}: {error}")
            log_failed(offset, error)
        else:
            cards = find_listing_cards(soup)

            if not cards:
                print(f"  ⚠️  No listings at offset {offset} — skipping.")
            else:
                records = []
                for card in cards:
                    parsed = parse_listing_card(card)
                    l_id = str(parsed.get('listing_id'))

                    if not l_id or not parsed.get('price_rm'):
                        continue

                    # 2. NEW: CHECK DUPLICATE BEFORE SCRAPING DETAILS
                    if l_id in scraped_ids:
                        skipped_count += 1
                        continue

                    listing_url = parsed.pop('_url')

                    # Visit detail page (Only for NEW listings)
                    time.sleep(random.uniform(1.5, 3.0))
                    detail = scrape_detail_page(http_session, listing_url) or {}

                    record = {
                        'listing_id':      l_id,
                        'property_type':   parsed['property_type'],
                        'description':     detail.get('description'),
                        'location':        parsed['location'],
                        'state':           parsed['state'],
                        'price_rm':        parsed['price_rm'],
                        'mortgage_est_rm': detail.get('mortgage_est_rm'),
                        'land_title':      detail.get('land_title'),
                        'size_sqft':       parsed['size_sqft'],
                        'bedrooms':        parsed['bedrooms'],
                        'bathrooms':       parsed['bathrooms'],
                        'tenure':          parsed['tenure'],
                        'seller_type':     parsed['seller_type'],
                        'url':             listing_url,
                        'img_url':         detail.get('img_url'),
                    }
                    records.append(record)
                    scraped_ids.add(l_id) # Add to set so we don't scrape it again in this run

                batch_records.extend(records)
                session_total    += len(records)
                batch_page_count += 1
                last_done_offset  = offset

                elapsed   = time.time() - state_start_time
                pages_done = i
                pages_left = len(session_offsets) - i
                avg_per_page = elapsed / pages_done
                eta_seconds  = avg_per_page * pages_left

                print(
                    f"  Page {i}/{len(session_offsets)} | "
                    f"offset {offset} | "
                    f"new: {len(records)} | "
                    f"skipped: {skipped_count} | "
                    f"total new: {session_total:,} | "
                    f"elapsed: {format_duration(elapsed)}"
                )

        # Flush to CSV every 10 pages
        if batch_page_count >= 10:
            append_to_csv(batch_records)
            batch_records    = []
            batch_page_count = 0

        time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))

except KeyboardInterrupt:
    print(f"\n⛔ Stopped manually.")

finally:
    if batch_records:
        append_to_csv(batch_records)

    total_elapsed = time.time() - state_start_time

    print(f"\n{'='*55}")
    print(f"  ✅ Session ended — {STATE.upper()}")
    print(f"  New records added     : {session_total:,}")
    print(f"  Duplicates skipped    : {skipped_count:,}")
    print(f"  Total execution time  : {format_duration(total_elapsed)}")
    print(f"{'='*55}")

ℹ️  Found 6,817 existing records in mudah_kedah.csv. These will be skipped.
State              : kedah
Pages this session : 77
Offsets            : 174 → 250
Est. time          : ~10.3 min

  Page 1/77 | offset 174 | new: 0 | skipped: 43 | total new: 0 | elapsed: 1s
  Page 2/77 | offset 175 | new: 12 | skipped: 74 | total new: 12 | elapsed: 46s
  Page 3/77 | offset 176 | new: 40 | skipped: 77 | total new: 52 | elapsed: 3m 30s
  Page 4/77 | offset 177 | new: 40 | skipped: 80 | total new: 92 | elapsed: 6m 12s
  Page 5/77 | offset 178 | new: 40 | skipped: 83 | total new: 132 | elapsed: 8m 51s
  Page 6/77 | offset 179 | new: 40 | skipped: 86 | total new: 172 | elapsed: 11m 36s
  Page 7/77 | offset 180 | new: 40 | skipped: 89 | total new: 212 | elapsed: 14m 20s
  Page 8/77 | offset 181 | new: 32 | skipped: 100 | total new: 244 | elapsed: 16m 30s
  Page 9/77 | offset 182 | new: 40 | skipped: 103 | total new: 284 | elapsed: 19m 15s
  Page 10/77 | offset 183 | new: 40 | skipped: 106 | total ne

## 🔁 Step 6 — Retry Failed Pages

In [ ]:
if not os.path.exists(FAILED_LOG):
    print("✅ No failed pages logged.")
else:
    with open(FAILED_LOG) as f:
        lines = [l.strip() for l in f if l.strip()]

    failed_offsets = []
    for line in lines:
        m = re.search(r'offset=(\d+)', line)
        if m:
            failed_offsets.append(int(m.group(1)))

    print(f"Found {len(failed_offsets)} failed offsets to retry: {failed_offsets}")

    retry_session = requests.Session()
    recovered, still_failed = [], []
    retry_start = time.time()

    for offset in failed_offsets:
        page_url = f"{BASE_URL}?o={offset}"
        soup, error = fetch_page(retry_session, page_url, retries=5)
        if soup:
            cards = find_listing_cards(soup)
            for card in cards:
                parsed = parse_listing_card(card)
                if not parsed.get('listing_id') or not parsed.get('price_rm'):
                    continue
                listing_url = parsed.pop('_url')
                time.sleep(random.uniform(1.5, 3.0))
                detail = scrape_detail_page(retry_session, listing_url)
                record = {
                    'listing_id':      parsed['listing_id'],
                    'property_type':   parsed['property_type'],
                    'description':     detail['description'],
                    'location':        parsed['location'],
                    'state':           parsed['state'],
                    'price_rm':        parsed['price_rm'],
                    'mortgage_est_rm': detail['mortgage_est_rm'],
                    'land_title':      detail['land_title'],
                    'size_sqft':       parsed['size_sqft'],
                    'bedrooms':        parsed['bedrooms'],
                    'bathrooms':       parsed['bathrooms'],
                    'tenure':          parsed['tenure'],
                    'seller_type':     parsed['seller_type'],
                    'seller_name':     detail['seller_name'],
                    'url':             listing_url,
                    'img_url':         detail['img_url'],
                }
                recovered.append(record)
            print(f"  ✅ Recovered offset {offset}: {len(cards)} listings")
        else:
            still_failed.append(offset)
            print(f"  ❌ Still failing: offset {offset}")
        time.sleep(random.uniform(4, 8))

    if recovered:
        append_to_csv(recovered)
        print(f"\n✅ Recovered {len(recovered)} listings from {len(failed_offsets) - len(still_failed)} pages")

    with open(FAILED_LOG, 'w') as f:
        for o in still_failed:
            f.write(f"offset={o}\tat={datetime.utcnow().isoformat()}\n")

    retry_elapsed = time.time() - retry_start
    print(f"  Retry execution time: {format_duration(retry_elapsed)}")

    if still_failed:
        print(f"  ⚠️  {len(still_failed)} offsets still failing: {still_failed}")
    else:
        print("  🎉 All failed pages recovered!")

✅ No failed pages logged.


## 📊 Step 7 — Preview & Validate Output

In [ ]:
if not os.path.exists(OUTPUT_CSV):
    print("No output file yet. Run Step 5 first.")
else:
    df = pd.read_csv(OUTPUT_CSV)
    df['price_rm'] = pd.to_numeric(df['price_rm'], errors='coerce')

    print(f"{'='*50}")
    print(f"  OUTPUT VALIDATION — {STATE.upper()}")
    print(f"{'='*50}")
    print(f"  Total rows          : {len(df):,}")
    print(f"  Unique listings     : {df['listing_id'].nunique():,}")
    print(f"  Duplicate rows      : {df.duplicated('listing_id').sum():,}")
    print(f"  Missing price       : {df['price_rm'].isna().sum():,}")
    print(f"  Missing location    : {df['location'].isna().sum():,}")
    print(f"  Has description     : {df['description'].notna().sum():,}")
    print(f"  Has seller name     : {df['seller_name'].notna().sum():,}")
    print(f"  Has land title      : {df['land_title'].notna().sum():,}")
    print(f"  Has mortgage est.   : {df['mortgage_est_rm'].notna().sum():,}")
    print(f"{'='*50}")

    display_cols = [c for c in COL_ORDER if c in df.columns]
    display(df[display_cols].head(10))

  OUTPUT VALIDATION — KEDAH
  Total rows          : 8,702
  Unique listings     : 8,702
  Duplicate rows      : 0
  Missing price       : 0
  Missing location    : 0
  Has description     : 8,700
  Has seller name     : 0
  Has land title      : 8,700
  Has mortgage est.   : 8,664


,listing_id,property_type,description,location,state,price_rm,mortgage_est_rm,land_title,size_sqft,bedrooms,bathrooms,tenure,seller_type,seller_name,url,img_url
0,114427893,Apartmentforsale,"The Laguna in Langkawi, luxurious apartment ne...","The Laguna Langkawi, Langkawi",Kedah,550000,2192.0,Non Bumi Lot,1050.0,2.0,2.0,Freehold,Private,NaN,https://www.mudah.my/the-laguna-in-langkawi-lu...,https://cdn.rnudah.com/images/plain/c5c106ec30...
1,114686703,New Terraced Houseforsale,Rumah Baru Siap Alor Setar - Taman Gamelan Fas...,Alor Setar,Kedah,318000,1267.0,Malay Reserved,1400.0,3.0,2.0,Freehold,Agent,NaN,https://www.mudah.my/rumah-baru-siap-alor-seta...,https://cdn.rnudah.com/images/plain/a78e2fc793...
2,112123333,New Terraced Houseforsale,Teres 1 Tingkat Baru Kota Sarang Semut CCC 202...,Kota Sarang Semut,Kedah,270000,1076.0,Non Bumi Lot,800.0,3.0,2.0,Freehold,Private,NaN,https://www.mudah.my/teres-1-tingkat-baru-kota...,https://cdn.rnudah.com/images/plain/a725ac9427...
3,112065125,1-storey Terraced Houseforsale,"CORNER LOT Teres setingkat Taman Selasih, Kuli...",Kulim,Kedah,350000,1395.0,Bumi Lot,2690.0,3.0,2.0,Freehold,Agent,NaN,https://www.mudah.my/corner-lot-teres-setingka...,https://cdn.rnudah.com/images/plain/c7fa9cd90d...
4,114679006,Semi-Detached Houseforsale,"MURAH‼️Semi D Single Storey, Astana Park Home ...",Sungai Petani,Kedah,435000,1733.0,Non Bumi Lot,3200.0,4.0,2.0,Freehold,Agent,NaN,https://www.mudah.my/murah-semi-d-single-store...,https://cdn.rnudah.com/images/plain/7d63d109c4...
5,108571317,1-storey Terraced Houseforsale,For Sale|Single Storey Terrace|Taman Kempas|Su...,Sungai Petani,Kedah,245000,976.0,Non Bumi Lot,1240.0,3.0,2.0,Freehold,Agent,NaN,https://www.mudah.my/for-sale-single-storey-te...,https://cdn.rnudah.com/images/plain/7e1ed4dcf9...
6,114742125,2-storey Terraced Houseforsale,2 Storey Terrace End Lot With Spacious (big)La...,Alor Setar,Kedah,540000,2152.0,Non Bumi Lot,3121.0,4.0,2.0,Freehold,Agent,NaN,https://www.mudah.my/2-storey-terrace-end-lot-...,https://cdn.rnudah.com/images/plain/37b48733b4...
7,113266123,2-storey Terraced Houseforsale,Rumah Baru Aspera Elite 2 - Bukit Banyan - Gat...,Sungai Petani,Kedah,457622,1824.0,Non Bumi Lot,1871.0,4.0,4.0,Freehold,Agent,NaN,https://www.mudah.my/rumah-baru-aspera-elite-2...,https://cdn.rnudah.com/images/plain/4bee2b6727...
8,114742170,Bungalow Houseforsale,For Sale|Double Storey Bungalow|Bandar Utama S...,Sungai Petani,Kedah,600000,2391.0,Non Bumi Lot,3660.0,5.0,4.0,Freehold,Agent,NaN,https://www.mudah.my/for-sale-double-storey-bu...,https://cdn.rnudah.com/images/plain/c85923e838...
9,114574950,1-storey Terraced Houseforsale,Zon Mawar Ambangan Height Sungai Petani\n✨ Rum...,Sungai Petani,Kedah,235000,936.0,Bumi Lot,1097.0,3.0,2.0,Freehold,Agent,NaN,https://www.mudah.my/zon-mawar-ambangan-height...,https://cdn.rnudah.com/images/plain/7ddb4f663f...


## ⬇️ Step 8 — Download Files

In [ ]:
from google.colab import files

for filepath in [OUTPUT_CSV, FAILED_LOG]:
    if os.path.exists(filepath):
        print(f"⬇️  Downloading {filepath}...")
        files.download(filepath)
    else:
        print(f"   (skipping {filepath} — not found)")

⬇️  Downloading mudah_kedah.csv...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

   (skipping mudah_kedah_failed.txt — not found)


## 🔗 Step 9 — Merge All State Files


In [ ]:
import pandas as pd
import os

CSV_FILES = [
    "/content/mudah_goe.csv",
    "/content/mudah_yasmin.csv",
    "/content/mudah_auni.csv"
]

OUTPUT_FILE = "/content/clean_data.csv"

dfs = []
for f in CSV_FILES:
    if os.path.exists(f):
        df_temp = pd.read_csv(f)
        print(f"  ✅ {os.path.basename(f)}: {len(df_temp):,} rows")
        dfs.append(df_temp)
    else:
        print(f"  ❌ Not found: {f}")

if dfs:
    df_all = pd.concat(dfs, ignore_index=True)
    before = len(df_all)

    df_all.drop_duplicates(subset=['listing_id'], keep='first', inplace=True)
    df_all.reset_index(drop=True, inplace=True)
    after = len(df_all)

    print(f"""
{'='*45}
  COMBINED RESULT
{'='*45}
  Total rows (combined)  : {before:,}
  Duplicates removed     : {before - after:,}
  Unique listings        : {after:,}
{'='*45}
""")

    df_all.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
    print(f"✅ Saved: {OUTPUT_FILE}")

    from google.colab import files
    files.download(OUTPUT_FILE)

  ❌ Not found: /content/mudah_goe.csv
  ❌ Not found: /content/mudah_yasmin.csv
  ❌ Not found: /content/mudah_auni.csv
